In [34]:
import pandas as pd

# ============================================================
# Input Data
# ============================================================

rows = [
    # ========================================================
    # Language Domain
    # ========================================================

    # ---------------- Factual1000 ----------------
    ["Language", "Factual1000", "GPT2-xs",    0.7203, 0.6670, 0.5780, 0.4101],
    ["Language", "Factual1000", "Pythia-1b",  0.6893, 0.6697, 0.4697, 0.3785],
    ["Language", "Factual1000", "Pythia-14m", 0.9562, 0.9562, 0.7247, 0.6944],
    ["Language", "Factual1000", "Average",    0.7886, 0.7643, 0.5908, 0.4943],

    # ---------------- LAMA ----------------
    ["Language", "LAMA", "GPT2-xs",    0.6995, 0.6527, 0.5578, 0.4021],
    ["Language", "LAMA", "Pythia-1b",  0.6578, 0.6311, 0.4435, 0.3467],
    ["Language", "LAMA", "Pythia-14m", 0.9452, 0.9443, 0.6965, 0.6621],
    ["Language", "LAMA", "Average",    0.7675, 0.7427, 0.5659, 0.4703],

    # ========================================================
    # Vision Domain (Placeholder)
    # ========================================================

    # ---------------- ImageNet ----------------
    ["Vision", "ImageNet", "ViT-tiny", "-", "-", "-", "-"],
    ["Vision", "ImageNet", "DeiT-tiny", "-", "-", "-", "-"],
    ["Vision", "ImageNet", "Average", "-", "-", "-", "-"],

    # ---------------- OfficeHome ----------------
    ["Vision", "OfficeHome", "ViT-tiny", "-", "-", "-", "-"],
    ["Vision", "OfficeHome", "DeiT-tiny", "-", "-", "-", "-"],
    ["Vision", "OfficeHome", "Average", "-", "-", "-", "-"],
]

columns = [
    "Domain",
    "Dataset",
    "Model",
    "FLOPs Original",
    "FLOPs Minimal",
    "Path Original",
    "Path Minimal",
]

df = pd.DataFrame(rows, columns=columns)

In [35]:
# ============================================================
# Utility
# ============================================================

def format_value(x):
    if isinstance(x, str):
        return x
    return f"{x:.4f}"

# ============================================================
# MAIN TEXT TABLE
# Domain-level averages across datasets
# ============================================================

agg_rows = []

for domain in df["Domain"].unique():

    domain_df = df[df["Domain"] == domain]

    models = domain_df["Model"].unique()

    for model in models:

        model_df = domain_df[domain_df["Model"] == model]

        numeric_df = model_df[
            model_df["FLOPs Original"] != "-"
        ]

        # Placeholder rows
        if len(numeric_df) == 0:

            agg_rows.append([
                domain,
                model,
                "-",
                "-",
                "-",
                "-"
            ])

            continue

        agg_rows.append([
            domain,
            model,
            numeric_df["FLOPs Original"].mean(),
            numeric_df["FLOPs Minimal"].mean(),
            numeric_df["Path Original"].mean(),
            numeric_df["Path Minimal"].mean(),
        ])

main_df = pd.DataFrame(agg_rows, columns=[
    "Domain",
    "Model",
    "FLOPs Original",
    "FLOPs Minimal",
    "Path Original",
    "Path Minimal",
])

In [36]:
# ============================================================
# Generate MAIN TABLE (Arrow + Reduction Format)
# ============================================================

def reduction_str(orig, new):

    if orig == "-" or new == "-":
        return "-"

    reduction = (new - orig) / orig * 100

    return (
        f"{orig:.4f} $\\rightarrow$ {new:.4f} "
        f"({reduction:.1f}\\%)"
    )

main_latex = []

main_latex.append(r"\begin{table}[t]")
main_latex.append(r"\centering")

main_latex.append(r"\caption{")
main_latex.append(
    r"Average sparsity before and after applying the proposed sparsity optimization algorithm. "
    r"Lower values indicate better sparsity."
)
main_latex.append(r"}")

main_latex.append(r"\label{tab:sparsity_main}")

main_latex.append(r"\begin{tabular}{llcc}")

main_latex.append(r"\toprule")

main_latex.append(
    r"Domain & Model & FLOPs & Path \\"
)

main_latex.append(r"\midrule")

domains = main_df["Domain"].unique()

for domain_idx, domain in enumerate(domains):

    domain_df = main_df[main_df["Domain"] == domain]

    total_rows = len(domain_df)

    main_latex.append(
        rf"\multirow{{{total_rows}}}{{*}}{{{domain}}}"
    )

    rows_ = domain_df.values.tolist()

    for row_idx, row in enumerate(rows_):

        _, model, flops_o, flops_m, path_o, path_m = row

        flops_str = reduction_str(flops_o, flops_m)
        path_str  = reduction_str(path_o, path_m)

        if row_idx == 0:
            line = " "
        else:
            line = "& "

        line += (
            f"& {model} "
            f"& {flops_str} "
            f"& {path_str} \\\\"
        )

        main_latex.append(line)

    if domain_idx != len(domains) - 1:
        main_latex.append(r"\midrule")

main_latex.append(r"\bottomrule")
main_latex.append(r"\end{tabular}")
main_latex.append(r"\end{table}")

main_latex_code = "\n".join(main_latex)

print(main_latex_code)

\begin{table}[t]
\centering
\caption{
Average sparsity before and after applying the proposed sparsity optimization algorithm. Lower values indicate better sparsity.
}
\label{tab:sparsity_main}
\begin{tabular}{llcc}
\toprule
Domain & Model & FLOPs & Path \\
\midrule
\multirow{4}{*}{Language}
 & GPT2-xs & 0.7099 $\rightarrow$ 0.6599 (-7.1\%) & 0.5679 $\rightarrow$ 0.4061 (-28.5\%) \\
& & Pythia-1b & 0.6736 $\rightarrow$ 0.6504 (-3.4\%) & 0.4566 $\rightarrow$ 0.3626 (-20.6\%) \\
& & Pythia-14m & 0.9507 $\rightarrow$ 0.9503 (-0.0\%) & 0.7106 $\rightarrow$ 0.6783 (-4.6\%) \\
& & Average & 0.7780 $\rightarrow$ 0.7535 (-3.2\%) & 0.5783 $\rightarrow$ 0.4823 (-16.6\%) \\
\midrule
\multirow{3}{*}{Vision}
 & ViT-tiny & - & - \\
& & DeiT-tiny & - & - \\
& & Average & - & - \\
\bottomrule
\end{tabular}
\end{table}
